# BlurBall — Passo 1: testar os pesos pré-treinados no Parada4.mp4

**Objetivo (teste barato, ANTES de treinar):** correr o detetor de bola temporal **BlurBall**
(pré-treinado, sem fine-tune) no teu vídeo e ver **quanto sobe o recall da bola** face aos
**~72% atuais** do YOLO de frame único — sobretudo na bola rápida borrada (remates, serviços).

O que este notebook faz, por ordem:
1. Confirma que tens GPU.
2. Instala o BlurBall (repo oficial, licença MIT).
3. Descarrega os pesos pré-treinados (BlurBall + baselines).
4. Traz o `Parada4.mp4` para o Colab.
5. Corre a inferência (`step=1`, `threshold=0.7`) → gera `traj.csv` (bola por frame) + vídeo com a bola marcada.
6. Calcula a **taxa de deteção da bola** (proxy de recall) e compara com os 72%.

> **Antes de começar:** menu **Runtime → Change runtime type → GPU** (T4 chega).
> Corre as células **por ordem**, de cima para baixo.


### 1) Confirmar a GPU

In [ ]:
!nvidia-smi

### 2) Clonar o BlurBall e instalar as dependências

A instalação usa as versões exatas dos autores (reprodutível). Pode **pedir para reiniciar o
runtime** (aviso amarelo "RESTART RUNTIME"). Se pedir: clica em **Restart**, depois **volta a
correr esta célula** e continua para baixo — os ficheiros já clonados não se perdem.

In [ ]:
%cd /content
!test -d blurball || git clone https://github.com/cogsys-tuebingen/blurball.git
%cd /content/blurball
!pip install -q -r requirements.txt
print("\nInstalacao terminada.")

### 3) Descarregar os pesos pré-treinados

Vem tudo num único zip do Nextcloud dos autores (BlurBall + WASB + TrackNetV2 + outros).
**Nota:** os ficheiros de pesos NÃO têm extensão (chamam-se `blurball_best`, `wasb_endpoint_best`, ...),
por isso identificamo-los pelo tamanho (modelos = ficheiros grandes).

In [ ]:
import os, glob, shutil

# =============================================================================
#  PESOS DO BLURBALL — guardar cópia na TUA Drive
#
#  ⚠️ Vêm de um link partilhado de um servidor da Universidade de Tübingen:
#       cloud.cs.uni-tuebingen.de/.../download
#  Não é a Drive de um colega. É um link público de um grupo de investigação,
#  que pode mudar, expirar ou desaparecer sem aviso e sem ninguém a quem pedir.
#
#  E é o modelo que SUBSTITUIU o YOLO (recall 85.6% vs 67%). Se o link morrer,
#  não há como o refazer — os pesos são do paper deles, com dados que não temos.
#
#  Por isso: 1.ª vez -> descarrega e GUARDA na tua Drive.
#            depois  -> usa a tua cópia. Nunca mais depende deles.
# =============================================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_PESOS = '/content/drive/MyDrive/PadelPro/pesos/blurball'
os.makedirs(DRIVE_PESOS, exist_ok=True)
os.makedirs('/content/weights', exist_ok=True)

ja_guardados = [p for p in glob.glob(DRIVE_PESOS + '/**/*', recursive=True)
                if os.path.isfile(p) and os.path.getsize(p) > 1_000_000]

if ja_guardados:
    print(f'Pesos vindos da TUA Drive ({len(ja_guardados)} ficheiros).')
    for src in ja_guardados:
        dst = '/content/weights/' + os.path.basename(src)
        shutil.copy(src, dst)
else:
    print('1.a vez: a descarregar de Tubingen...')
    !wget -q --show-progress -O /content/weights.zip "https://cloud.cs.uni-tuebingen.de/index.php/s/6Z8TpM3sXRKHzGC/download"
    !cd /content/weights && unzip -o -q /content/weights.zip
    achados = [p for p in glob.glob('/content/weights/**/*', recursive=True)
               if os.path.isfile(p) and os.path.getsize(p) > 1_000_000]
    for src in achados:
        shutil.copy(src, os.path.join(DRIVE_PESOS, os.path.basename(src)))
    print(f'\n*** COPIA GUARDADA em {DRIVE_PESOS} ({len(achados)} ficheiros) ***')
    print('    A partir de agora usa-se esta. Ja nao dependemos de Tubingen.')

# os pesos NAO tem extensao -> apanhar pelos ficheiros grandes (>1 MB)
todos = sorted(set(p for p in glob.glob('/content/weights/**/*', recursive=True)
                   if os.path.isfile(p) and os.path.getsize(p) > 1_000_000))
print('\nFicheiros de pesos disponiveis:')
for t in todos:
    print(f'   {os.path.basename(t):30s} {os.path.getsize(t)/1e6:6.1f} MB')

# escolher automaticamente o do BlurBall
blur = [t for t in todos if 'blurball' in os.path.basename(t).lower()]
BLURBALL_W = blur[0] if blur else (todos[0] if todos else None)
assert BLURBALL_W, "Nao encontrei pesos — verifica o download acima."
print("\n>>> Vou usar como BlurBall:", BLURBALL_W)

> Se o ficheiro escolhido acima **não** for o do BlurBall, define à mão na célula seguinte
> (copia o caminho certo da lista impressa). Caso contrário, deixa como está.

In [ ]:
# BLURBALL_W = '/content/weights/.../blurball.pth.tar'   # <- descomenta e ajusta se preciso
print("BlurBall weights:", BLURBALL_W)

### 4) Apontar o repo para esta pasta (WASB_ROOT)

In [ ]:
!sed -i 's|^WASB_ROOT:.*|WASB_ROOT: /content/blurball|' /content/blurball/src/configs/global.yaml
!cat /content/blurball/src/configs/global.yaml
import os; os.environ['HYDRA_FULL_ERROR']='1'

### 5) Trazer o `Parada4.mp4` para o Colab

**Opção A — Google Drive (recomendada).** Põe o `Parada4.mp4` no teu Drive e ajusta o caminho.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
# AJUSTA este caminho ao local do video no teu Drive:
VIDEO_SRC = '/content/drive/MyDrive/Parada4.mp4'
VIDEO_PATH = '/content/Parada4.mp4'
shutil.copy(VIDEO_SRC, VIDEO_PATH)
print("Video pronto:", VIDEO_PATH, "=", os.path.getsize(VIDEO_PATH)//1024//1024, "MB")

**Opção B — Upload direto** (se não usares Drive). Descomenta e corre:

In [ ]:
# from google.colab import files
# up = files.upload()   # escolhe o Parada4.mp4
# import os
# VIDEO_PATH = '/content/' + list(up.keys())[0]
# print("Video:", VIDEO_PATH)

### (Opcional) Teste-relâmpago com 60s primeiro

Antes de correr o vídeo todo (~5 min de jogo → alguns minutos de inferência), podes validar
que está tudo a funcionar com só os primeiros 60s. Descomenta, corre, e faz a inferência.
Para o vídeo completo, **não** corras esta célula (ou volta a pôr `VIDEO_PATH` no ficheiro completo).

In [ ]:
# !ffmpeg -y -i /content/Parada4.mp4 -t 60 -c copy /content/Parada4_60s.mp4
# VIDEO_PATH = '/content/Parada4_60s.mp4'
# print("Agora VIDEO_PATH =", VIDEO_PATH)

### 6) Correr o BlurBall (step=1, threshold=0.7)

Primeiro extrai os frames do vídeo, depois corre o detetor frame a frame. Demora alguns minutos.
No fim escreve `traj.csv` (posição da bola por frame) e um vídeo com a bola marcada.

In [ ]:
# ⭐ THRESHOLD 0.7 -> 0.4
# A 0.7 so 46% dos frames tem bola. A 0.4 sobem para ~76%.
# As deteccoes que se ganham sao precisamente as DIFICEIS: bola borrada,
# rapida, pequena ao fundo -- os momentos que mais interessam (servico, remate).
# O lixo extra e limpo depois pelas regras: IMOVEIS + CONTINUIDADE.
# Recall pelo detetor, precisao pelas regras.

%cd /content/blurball/src
!python main.py --config-name=inference_blurball \
    detector.model_path="{BLURBALL_W}" \
    +input_vid="{VIDEO_PATH}" \
    detector.step=1 \
    detector.postprocessor.score_threshold=0.4 \
    runner.vis_hm=false

### 7) Localizar os resultados

In [ ]:
import os
vdir  = os.path.dirname(VIDEO_PATH)
vname = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
TRAJ    = os.path.join(vdir, f'frames_{vname}', 'traj.csv')   # bola por frame
OVERLAY = os.path.join(vdir, 'frames.mp4')                    # video com a bola marcada
print("traj.csv :", TRAJ,    "->", os.path.exists(TRAJ))
print("overlay  :", OVERLAY, "->", os.path.exists(OVERLAY))

### 8) Métrica barata — taxa de deteção da bola (proxy de recall) vs 72%

Não temos ground-truth da bola frame-a-frame, mas temos os **12 rallies reais** (117s de jogo).
Dentro desses intervalos a bola está **sempre em jogo**, por isso a **% de frames com bola detetada
dentro dos rallies** é um bom proxy do recall — e é **comparável aos ~72%** do YOLO atual.

(O `traj.csv` traz `Visibility`=1 quando o BlurBall deteta bola nesse frame.)

In [ ]:
import pandas as pd, cv2, os

BASELINE_YOLO = 0.72   # recall da bola do YOLO de frame unico (referencia atual)

# 12 rallies reais do Parada4 (segundos) — ground_truth_parada4.md
GT = [(38.0,41.5),(46.8,67.5),(77.6,85.5),(95.9,111.1),(122.4,135.9),(157.9,169.4),
      (178.1,186.5),(197.0,202.1),(210.5,216.3),(229.9,237.3),(249.6,255.0),(263.8,276.4)]

df = pd.read_csv(TRAJ)
N  = len(df)
fps = cv2.VideoCapture(VIDEO_PATH).get(cv2.CAP_PROP_FPS) or 29.97
vis = df['Visibility'].values

det_global = vis.mean()

# uniao das janelas de jogo
in_play = [False]*N
per_rally = []
for k,(a,b) in enumerate(GT, start=1):
    f0, f1 = int(round(a*fps)), min(N-1, int(round(b*fps)))
    seg = vis[f0:f1+1]
    for i in range(f0, f1+1):
        if i < N: in_play[i] = True
    per_rally.append((k, round(a,1), round(b,1), len(seg), float(seg.mean()) if len(seg) else 0.0))

import numpy as np
mask = np.array(in_play)
det_inplay = vis[mask].mean() if mask.any() else 0.0

print(f"Frames totais no video      : {N}  (fps={fps:.2f})")
print(f"Deteção GLOBAL da bola      : {det_global*100:5.1f}%  (inclui pausas, so' referencia)")
print(f"Deteção DENTRO dos rallies  : {det_inplay*100:5.1f}%   <-- comparar com YOLO {BASELINE_YOLO*100:.0f}%")
delta = (det_inplay - BASELINE_YOLO)*100
print(f"Δ vs YOLO frame-unico       : {delta:+.1f} pontos percentuais")
print("\nPor rally (dentro-do-jogo):")
print(f"{'#':>2} {'ini':>6} {'fim':>6} {'frames':>7} {'det%':>7}")
for k,a,b,nf,d in per_rally:
    print(f"{k:>2} {a:>6} {b:>6} {nf:>7} {d*100:6.1f}%")

# guardar um pequeno relatorio
rep = pd.DataFrame(per_rally, columns=['rally','ini_s','fim_s','frames','det_rate'])
rep.to_csv('/content/BlurBall_recall_por_rally.csv', index=False)
print("\nRelatorio: /content/BlurBall_recall_por_rally.csv")

### 9) Ver o vídeo com a bola marcada (olhar aos serviços e remates)

O sítio onde o YOLO falha é a bola rápida borrada. Vê o overlay e repara se o BlurBall
apanha a bola nos **serviços** e **remates**. Guardamos também os resultados no Drive.

In [ ]:
import shutil
# guardar no Drive (se montaste o Drive no passo 5A)
try:
    shutil.copy(TRAJ,    '/content/drive/MyDrive/BlurBall_traj_Parada4.csv')
    shutil.copy(OVERLAY, '/content/drive/MyDrive/BlurBall_overlay_Parada4.mp4')
    shutil.copy('/content/BlurBall_recall_por_rally.csv', '/content/drive/MyDrive/BlurBall_recall_por_rally.csv')
    print("Resultados guardados no teu Drive (MyDrive).")
except Exception as e:
    print("Nao guardei no Drive (montaste-o?):", e)

### O que decidir a seguir

- Se a **deteção dentro dos rallies** subir bem acima dos 72% (sobretudo nos serviços/remates
  no vídeo) → vale a pena o **Passo 2**: fine-tune com os teus frames Court-Master + treino no
  PadelTracker100.
- Depois, **Passo 3**: ligar o `traj.csv` (a coluna da bola) ao `rallies_bola.py` — o `ball_xy`
  entra tal e qual, **sem mudar as regras v9** — e medir contra o ground-truth (117s / 12 rallies).

> Nota: o `traj.csv` tem `X, Y` já em pixéis do vídeo original (960×540), `Visibility` (0/1) e,
> por ser BlurBall, também `L` e `Theta` (comprimento/orientação do risco de borrão).


## ⭐ Guardar com o nome certo

O ficheiro tem de se chamar **`traj_frames_Parada4_thr04.csv`** e ir para a pasta
**`dados_parada4/`** do projeto — e' de la' que o Centro de Decisoes o le.


In [ ]:
import glob, shutil, os

# procura o traj.csv que o BlurBall escreveu
cands = glob.glob('/content/blurball/**/traj*.csv', recursive=True)
cands += glob.glob('/content/**/traj*.csv', recursive=True)
cands = sorted(set(cands), key=os.path.getmtime, reverse=True)
assert cands, 'Nao encontrei nenhum traj.csv -- ve o output da celula anterior.'
src = cands[0]
print('encontrado:', src)

DST = '/content/traj_frames_Parada4_thr04.csv'
shutil.copy(src, DST)

# quantos frames tem bola?
import csv
rows = list(csv.DictReader(open(DST)))
vis = sum(1 for r in rows if r.get('Visibility') == '1')
print(f'\nframes: {len(rows)}')
print(f'com bola: {vis}  ({100*vis/len(rows):.1f}%)   <-- a thr=0.7 dava 46.2%')

from google.colab import files
files.download(DST)
print('\n>>> Poe o ficheiro em  dados_parada4/  e avisa o Centro de Decisoes.')
